# Benchmarking menedżerów haseł – normalizacja i ranking

Notatnik bierze surowy Excel `menedżer haseł-2.xlsx` (10 menedżerów × 15 kryteriów) i:
1. sprowadza wszystkie wartości do skali 0–1 (normalizacja min-max),
2. liczy sumę ważoną (0–100) dla każdego menedżera,
3. układa ranking i wskazuje lidera i najsłabszego,
4. dopisuje to wszystko do Excela razem z wykresem i zapisuje nowy plik `menedzer_hasel_wyniki.xlsx`.

**Jak uruchomić:** *Środowisko wykonawcze → Uruchom wszystko*. W Colabie notatnik poprosi o wgranie Excela.
Lokalnie wystarczy, że Excel leży w tym samym folderze.

## 1. Wczytanie Excela

In [1]:
import os
import pandas as pd
from openpyxl import load_workbook
from openpyxl.chart import BarChart, Reference
from openpyxl.chart.label import DataLabelList
from openpyxl.chart.layout import Layout, ManualLayout
from copy import copy

NAZWA_EXCELA = "menedżer haseł-2.xlsx"

# lokalnie: plik leży obok notatnika; w Colabie: trzeba go wgrać
if os.path.exists(NAZWA_EXCELA):
    nazwa_pliku = NAZWA_EXCELA
else:
    from google.colab import files
    wgrane = files.upload()
    nazwa_pliku = list(wgrane.keys())[0]

wb = load_workbook(nazwa_pliku)
ws = wb["Arkusz1"]

### Gdzie co jest w arkuszu
Wiersz 2 – nazwy kryteriów, wiersz 3 – S/D, wiersz 4 – wagi, wiersze 5–14 – menedżerowie, kolumny B–P – kryteria.
Jeśli ktoś przesunie tabelę w Excelu, wystarczy poprawić liczby poniżej.

In [2]:
WIERSZ_NAGLOWKA = 2     # nazwy kryteriów
WIERSZ_SD = 3           # litery S / D
WIERSZ_WAG = 4          # wagi
PIERWSZY_MENEDZER = 5   # pierwszy menedżer
OSTATNI_MENEDZER = 14   # ostatni menedżer
PIERWSZA_KOLUMNA = 2    # kolumna B
OSTATNIA_KOLUMNA = 16   # kolumna P
PRZESUNIECIE = 16       # o ile wierszy niżej wstawimy nową tabelę

menedzerowie = range(PIERWSZY_MENEDZER, OSTATNI_MENEDZER + 1)
kryteria = range(PIERWSZA_KOLUMNA, OSTATNIA_KOLUMNA + 1)

# podgląd danych jako tabela
przed = pd.DataFrame(
    [[ws.cell(w, k).value for k in kryteria] for w in menedzerowie],
    index=[ws.cell(w, 1).value for w in menedzerowie],
    columns=[ws.cell(WIERSZ_NAGLOWKA, k).value for k in kryteria],
)
przed

,Darmowa wersja,Cena planu płatnego (rocznie),Ilość platform obsługujących,Monitoring wycieków,Szyfrowanie end-to-end,Historia wycieków,Open source,Niezależne audyty bezpieczeństwa,Łatwość przenoszenia danych do innego menedżera,Udostępnianie haseł innym osobom,Limit urządzeń w darmowej wersji,Passkeys,Odblokowanie biometrią,Dwuetapowa weryfikacja,Maskowanie adresu e-mail
Apple Hasła (Apple Passwords / pęk kluczy iCloud),TAK,Brak planu płatnego,3,TAK,TAK,1,NIE,2,4,4,5,4,TAK,TAK,TAK
Menedżer haseł Google (Google Password Manager),TAK,Brak planu płatnego,3,TAK,NIE,1,NIE,2,3,3,5,5,TAK,TAK,NIE
1Password,NIE,"47,88 USD",6,NIE,TAK,3,NIE,5,5,1,1,1,TAK,TAK,NIE
Bitwarden,TAK,"19,80 USD",6,TAK,TAK,2,TAK,5,5,2,5,5,TAK,TAK,NIE
LastPass,TAK,"36,00 USD",5,TAK,TAK,5,NIE,3,3,2,3,3,TAK,TAK,NIE
Dashlane,NIE,"59,88 USD",4,NIE,TAK,2,NIE,3,4,1,1,1,TAK,TAK,NIE
NordPass,TAK,"23,88 USD",6,NIE,TAK,1,NIE,4,3,1,3,4,TAK,TAK,TAK
Proton Pass,TAK,"35,88 USD",6,NIE,TAK,1,TAK,4,4,5,5,5,TAK,TAK,TAK
Keeper,TAK,"42,96 USD",6,NIE,TAK,1,NIE,4,4,1,1,2,TAK,TAK,NIE
Microsoft Authenticator / Menedżer haseł w Edge,TAK,Brak planu płatnego,3,TAK,NIE,1,NIE,2,3,1,5,3,TAK,TAK,NIE


## 2. Normalizacja (wszystko na skalę 0–1)

- **TAK/NIE** → TAK = 1, NIE = 0 (dla destymulanty odwrotnie).
- **Liczby i skala 1–5** → wzór min-max:
  - stymulanta (im więcej, tym lepiej): `(x − min) / (max − min)`
  - destymulanta (im mniej, tym lepiej): `(max − x) / (max − min)`
- **„Brak planu płatnego”** w cenie → 1, czyli najlepiej (nie trzeba nic płacić).

In [3]:
# zamiana "47,88 USD" na liczbę 47.88
def na_liczbe(wartosc):
    if isinstance(wartosc, (int, float)):
        return wartosc
    tekst = str(wartosc).replace("USD", "").replace(",", ".").strip()
    return float(tekst)


wyniki = {}   # wyniki[(wiersz, kolumna)] = wynik 0-1

for kol in kryteria:
    nazwa = ws.cell(WIERSZ_NAGLOWKA, kol).value
    typ = ws.cell(WIERSZ_SD, kol).value          # "S" albo "D"
    wartosci = [ws.cell(w, kol).value for w in menedzerowie]

    # puste komórki zatrzymują program
    if None in wartosci:
        raise ValueError("Pusta komórka w kryterium: " + str(nazwa) + ". Uzupełnij ją w Excelu.")

    # TAK/NIE
    if all(w in ("TAK", "NIE") for w in wartosci):
        for w in menedzerowie:
            if ws.cell(w, kol).value == "TAK":
                wynik = 1
            else:
                wynik = 0
            if typ == "D":
                wynik = 1 - wynik
            wyniki[(w, kol)] = wynik

    # skala 1-5 i liczby: wzór min-max
    else:
        # do min i max bierzemy tylko liczby (bez "Brak planu płatnego")
        liczby = []
        for v in wartosci:
            if not str(v).startswith("Brak"):
                liczby.append(na_liczbe(v))
        minimum = min(liczby)
        maksimum = max(liczby)
        print(nazwa, "-> min =", minimum, ", max =", maksimum)

        if maksimum == minimum:
            raise ValueError("W kryterium " + str(nazwa) + " wszystkie liczby są takie same, nie da się policzyć wzoru.")

        for w in menedzerowie:
            v = ws.cell(w, kol).value
            if str(v).startswith("Brak"):      # "Brak planu płatnego" = 1
                wynik = 1
            else:
                x = na_liczbe(v)
                if typ == "S":                 # stymulanta
                    wynik = (x - minimum) / (maksimum - minimum)
                else:                          # destymulanta
                    wynik = (maksimum - x) / (maksimum - minimum)
            wyniki[(w, kol)] = wynik

# podgląd po normalizacji
po = pd.DataFrame(
    [[wyniki[(w, k)] for k in kryteria] for w in menedzerowie],
    index=przed.index,
    columns=przed.columns,
)
po.round(2)

Cena planu płatnego (rocznie) -> min = 19.8 , max = 59.88
Ilość platform obsługujących -> min = 3 , max = 6
Historia wycieków -> min = 1 , max = 5
Niezależne audyty bezpieczeństwa -> min = 2 , max = 5
Łatwość przenoszenia danych do innego menedżera -> min = 3 , max = 5
Udostępnianie haseł innym osobom -> min = 1 , max = 5
Limit urządzeń w darmowej wersji -> min = 1 , max = 5
Passkeys -> min = 1 , max = 5


,Darmowa wersja,Cena planu płatnego (rocznie),Ilość platform obsługujących,Monitoring wycieków,Szyfrowanie end-to-end,Historia wycieków,Open source,Niezależne audyty bezpieczeństwa,Łatwość przenoszenia danych do innego menedżera,Udostępnianie haseł innym osobom,Limit urządzeń w darmowej wersji,Passkeys,Odblokowanie biometrią,Dwuetapowa weryfikacja,Maskowanie adresu e-mail
Apple Hasła (Apple Passwords / pęk kluczy iCloud),1,1.00,0.00,1,1,1.00,0,0.00,0.5,0.75,1.0,0.75,1,1,1
Menedżer haseł Google (Google Password Manager),1,1.00,0.00,1,0,1.00,0,0.00,0.0,0.50,1.0,1.00,1,1,0
1Password,0,0.30,1.00,0,1,0.50,0,1.00,1.0,0.00,0.0,0.00,1,1,0
Bitwarden,1,1.00,1.00,1,1,0.75,1,1.00,1.0,0.25,1.0,1.00,1,1,0
LastPass,1,0.60,0.67,1,1,0.00,0,0.33,0.0,0.25,0.5,0.50,1,1,0
Dashlane,0,0.00,0.33,0,1,0.75,0,0.33,0.5,0.00,0.0,0.00,1,1,0
NordPass,1,0.90,1.00,0,1,1.00,0,0.67,0.0,0.00,0.5,0.75,1,1,1
Proton Pass,1,0.60,1.00,0,1,1.00,1,0.67,0.5,1.00,1.0,1.00,1,1,1
Keeper,1,0.42,1.00,0,1,1.00,0,0.67,0.5,0.00,0.0,0.25,1,1,0
Microsoft Authenticator / Menedżer haseł w Edge,1,1.00,0.00,1,0,1.00,0,0.00,0.0,0.00,1.0,0.50,1,1,0


### Wpisanie tabeli po normalizacji do Excela (16 wierszy niżej, z takim samym formatowaniem)

In [4]:
ws.cell(WIERSZ_NAGLOWKA - 1 + PRZESUNIECIE, 1).value = "DANE PO NORMALIZACJI ([0,1])"

# nagłówki: nazwy kryteriów, S/D, wagi
for wiersz in (WIERSZ_NAGLOWKA, WIERSZ_SD, WIERSZ_WAG):
    for kol in range(1, OSTATNIA_KOLUMNA + 1):
        stara = ws.cell(wiersz, kol)
        nowa = ws.cell(wiersz + PRZESUNIECIE, kol)
        nowa.value = stara.value
        nowa._style = copy(stara._style)

# menedżerowie i ich wyniki 0-1
for w in menedzerowie:
    stara = ws.cell(w, 1)
    nowa = ws.cell(w + PRZESUNIECIE, 1)
    nowa.value = stara.value
    nowa._style = copy(stara._style)
    for kol in kryteria:
        stara = ws.cell(w, kol)
        nowa = ws.cell(w + PRZESUNIECIE, kol)
        nowa.value = wyniki[(w, kol)]
        nowa._style = copy(stara._style)
        nowa.number_format = "0.00"

## 3. Suma ważona (0–100)

Dla każdego menedżera: `W = (waga₁·wynik₁ + waga₂·wynik₂ + … ) × 100`. Wagi sumują się do 1, więc maksimum to 100.

In [5]:
# kontrola: wagi powinny sumować się do 1
print("Suma wag:", round(sum(ws.cell(WIERSZ_WAG, k).value for k in kryteria), 3))

KOLUMNA_SUMY = OSTATNIA_KOLUMNA + 1    # kolumna Q

# nagłówek kolumny z sumą (taki sam wygląd jak pozostałe nagłówki)
naglowek = ws.cell(WIERSZ_NAGLOWKA + PRZESUNIECIE, KOLUMNA_SUMY)
naglowek.value = "Suma ważona (0-100)"
naglowek._style = copy(ws.cell(WIERSZ_NAGLOWKA + PRZESUNIECIE, OSTATNIA_KOLUMNA)._style)

sumy = {}   # sumy[krótka nazwa menedżera] = suma ważona
for w in menedzerowie:
    suma = 0
    for kol in kryteria:
        waga = ws.cell(WIERSZ_WAG, kol).value
        suma = suma + wyniki[(w, kol)] * waga
    suma = round(suma * 100, 1)
    komorka = ws.cell(w + PRZESUNIECIE, KOLUMNA_SUMY)
    komorka.value = suma
    komorka._style = copy(ws.cell(w + PRZESUNIECIE, OSTATNIA_KOLUMNA)._style)
    # krótka nazwa (bez nawiasów i dopisków po ukośniku), np. "Apple Hasła"
    krotka = str(ws.cell(w, 1).value).split(" (")[0].split(" /")[0].strip()
    sumy[krotka] = suma

ws.column_dimensions["Q"].width = len("Suma ważona (0-100)") + 2

Suma wag: 1.0


## 4. Ranking, lider i najsłabszy

In [6]:
# od największej sumy do najmniejszej
ranking = sorted(sumy.items(), key=lambda para: para[1], reverse=True)

START = OSTATNI_MENEDZER + PRZESUNIECIE + 3     # wiersz, od którego zaczyna się ranking w Excelu
ws.cell(START, 1).value = "RANKING MENEDŻERÓW"
ws.cell(START + 1, 1).value = "Menedżer"
ws.cell(START + 1, 2).value = "Suma ważona (0-100)"

wiersz = START + 2
for nazwa, suma in ranking:
    ws.cell(wiersz, 1).value = nazwa
    ws.cell(wiersz, 2).value = suma
    wiersz = wiersz + 1
ostatni_wiersz = wiersz - 1

najwyzsza = ranking[0][1]
najnizsza = ranking[-1][1]
liderzy = [n for n, s in ranking if s == najwyzsza]
najslabsi = [n for n, s in ranking if s == najnizsza]

ws.cell(ostatni_wiersz + 2, 1).value = "Lider (" + str(najwyzsza) + "): " + ", ".join(liderzy)
ws.cell(ostatni_wiersz + 3, 1).value = "Najsłabszy (" + str(najnizsza) + "): " + ", ".join(najslabsi)
print("Lider:", liderzy, "-", najwyzsza)
print("Najsłabszy:", najslabsi, "-", najnizsza)

pd.DataFrame(ranking, columns=["Menedżer", "Suma ważona (0-100)"], index=range(1, len(ranking) + 1))

Lider: ['Bitwarden'] - 90.0
Najsłabszy: ['Dashlane'] - 37.2


,Menedżer,Suma ważona (0-100)
1,Bitwarden,90.0
2,Proton Pass,84.2
3,Apple Hasła,73.3
4,NordPass,71.9
5,Keeper,60.6
6,Menedżer haseł Google,55.0
7,LastPass,53.0
8,1Password,51.0
9,Microsoft Authenticator,50.5
10,Dashlane,37.2


## 5. Wykres rankingu w Excelu

In [7]:
wykres = BarChart()
wykres.type = "bar"                            # słupki poziome
wykres.title = "Ranking menedżerów haseł (wynik W, 0-100)"
wykres.title.overlay = False                   # tytuł nad wykresem, nie na słupkach
wykres.legend = None                           # jedna seria, legenda niepotrzebna

dane = Reference(ws, min_col=2, min_row=START + 1, max_row=ostatni_wiersz)     # sumy (z nagłówkiem)
nazwy = Reference(ws, min_col=1, min_row=START + 2, max_row=ostatni_wiersz)    # nazwy menedżerów
wykres.add_data(dane, titles_from_data=True)
wykres.set_categories(nazwy)

# wygląd osi
wykres.x_axis.scaling.orientation = "maxMin"   # lider na górze
wykres.y_axis.crosses = "max"                  # skala 0-100 na dole wykresu
wykres.x_axis.delete = False                   # pokaż oś z nazwami
wykres.y_axis.delete = False                   # pokaż oś z liczbami
wykres.y_axis.scaling.min = 0
wykres.y_axis.scaling.max = 100

# liczby na końcu słupków (tylko wartość, bez reszty opisów)
wykres.dataLabels = DataLabelList()
wykres.dataLabels.showVal = True
wykres.dataLabels.showSerName = False
wykres.dataLabels.showCatName = False
wykres.dataLabels.showLegendKey = False
wykres.dataLabels.showPercent = False
wykres.dataLabels.position = "outEnd"

# rozmiar i położenie
wykres.gapWidth = 60                           # grubsze słupki
wykres.plot_area.layout = Layout(manualLayout=ManualLayout(xMode="edge", yMode="edge", x=0.22, y=0.16, w=0.72, h=0.72))  # miejsce na nazwy po lewej i tytuł u góry
wykres.width = 24
wykres.height = 14

ws.add_chart(wykres, "D" + str(START))

## 6. Zapis nowego Excela

In [8]:
nowy_plik = "menedzer_hasel_wyniki.xlsx"
wb.save(nowy_plik)

# w Colabie plik od razu się pobiera, lokalnie zostaje w folderze
try:
    from google.colab import files
    files.download(nowy_plik)
except ImportError:
    pass
print("Gotowe! Zapisano:", nowy_plik)

Gotowe! Zapisano: menedzer_hasel_wyniki.xlsx


# 7. Dashboard

Interaktywne wykresy (plotly): po najechaniu myszką widać dokładne wartości. Wszystko działa w Colabie bez instalowania czegokolwiek.

In [9]:
import plotly.express as px

# krótkie nazwy menedżerów zamiast długich z Excela
po_krotko = po.copy()
po_krotko.index = list(sumy.keys())

lider = liderzy[0]
najslabszy = najslabsi[0]

## 7.1 Ranking

In [10]:
nazwy = [n for n, s in ranking]
wartosci = [s for n, s in ranking]

# kolor: zielony lider, czerwony najsłabszy, szary reszta
kolory = []
for n in nazwy:
    if n in liderzy:
        kolory.append("lider")
    elif n in najslabsi:
        kolory.append("najsłabszy")
    else:
        kolory.append("pozostali")

fig = px.bar(
    x=wartosci, y=nazwy, color=kolory, orientation="h", text=wartosci,
    color_discrete_map={"lider": "#2e8b57", "najsłabszy": "#c0392b", "pozostali": "#9aa5b1"},
    labels={"x": "Suma ważona (0-100)", "y": "", "color": ""},
    title="Ranking menedżerów haseł",
)
fig.update_yaxes(categoryorder="array", categoryarray=nazwy[::-1])   # lider na górze
fig.update_xaxes(range=[0, 100])
fig.show()

## 7.2 Mapa wyników po normalizacji

Każde pole to wynik 0–1 menedżera w danym kryterium: zielone = najlepiej, czerwone = najgorzej.
Wiersze są ułożone tak jak w rankingu.

In [11]:
fig = px.imshow(
    po_krotko.loc[nazwy].round(2),
    color_continuous_scale="RdYlGn", zmin=0, zmax=1, text_auto=True, aspect="auto",
    title="Wyniki 0-1 w każdym kryterium",
)
fig.update_layout(height=550)
fig.update_xaxes(tickangle=45)
fig.show()

## 7.3 Skąd bierze się różnica w punktach?

Wkład kryterium = waga × wynik 0–1 × 100. Suma wkładów daje wynik z rankingu.
Domyślnie porównujemy lidera z najsłabszym. Żeby porównać innych, wpisz ich nazwy (np. `"LastPass"`) i uruchom komórkę jeszcze raz.

In [12]:
MENEDZER_A = lider          # np. "Bitwarden"
MENEDZER_B = najslabszy     # np. "LastPass"

print("Do wyboru:", ", ".join(sumy.keys()))

wagi = [ws.cell(WIERSZ_WAG, k).value for k in kryteria]

tabela = pd.DataFrame({
    MENEDZER_A: po_krotko.loc[MENEDZER_A] * wagi * 100,
    MENEDZER_B: po_krotko.loc[MENEDZER_B] * wagi * 100,
})

fig = px.bar(
    tabela, x=tabela.index, y=[MENEDZER_A, MENEDZER_B], barmode="group",
    labels={"value": "punkty", "x": "", "variable": ""},
    title=MENEDZER_A + " (" + str(sumy[MENEDZER_A]) + " pkt) vs " + MENEDZER_B + " (" + str(sumy[MENEDZER_B]) + " pkt)",
)
fig.update_xaxes(tickangle=45)
fig.show()

# kryteria, w których różnica jest największa
tabela["różnica"] = tabela[MENEDZER_A] - tabela[MENEDZER_B]
tabela.round(1).sort_values("różnica", key=abs, ascending=False).head(5)

Do wyboru: Apple Hasła, Menedżer haseł Google, 1Password, Bitwarden, LastPass, Dashlane, NordPass, Proton Pass, Keeper, Microsoft Authenticator


,Bitwarden,Dashlane,różnica
Cena planu płatnego (rocznie),10.0,0.0,10.0
Darmowa wersja,7.0,0.0,7.0
Niezależne audyty bezpieczeństwa,10.0,3.3,6.7
Monitoring wycieków,6.0,0.0,6.0
Open source,5.0,0.0,5.0
